Загрузка и валидация

In [ ]:
# Cell 1: setup + load
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("students.csv")  
OUT_DIR   = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

df_raw = pd.read_csv(DATA_PATH, encoding="utf-8-sig", dtype=str)
df = df_raw.copy()

# Типы: EventIndex — число, Amount — float, Timestamp — tz-aware datetime
df["EventIndex"] = pd.to_numeric(df["EventIndex"], errors="raise").astype(int)
df["Amount"]     = pd.to_numeric(df["Amount"],     errors="raise")
df["Timestamp"]  = pd.to_datetime(df["Timestamp"], format="ISO8601", errors="raise")

# Пустые необязательные поля приводим к пустой строке (чтобы не путать NaN и "")
for col in ["TaskId", "UserId"]:
    df[col] = df[col].fillna("").str.strip()

# Основная сортировка: внутри экземпляра — по числовому EventIndex
df = (df.sort_values(["ProcessInstanceId", "EventIndex"], kind="mergesort")
        .reset_index(drop=True))

print(f"Строк:        {len(df)}")
print(f"Колонок:      {df.shape[1]}")
print(f"Экземпляров:  {df['ProcessInstanceId'].nunique()}")
print(f"Тип Timestamp: {df['Timestamp'].dtype}")
df.head(3)

In [ ]:
# Cell 2: validation report
report = {}     # сводка
issues = []     # найденные проблемы

# 1. Уникальность EventId
dup_eventid = int(df["EventId"].duplicated().sum())
report["unique_event_id"] = df["EventId"].nunique()
if dup_eventid:
    issues.append(f"Дубли EventId: {dup_eventid}")

# 2. Внутри экземпляра EventIndex — это 1..N без пропусков
ok_idx = (
    df.groupby("ProcessInstanceId")["EventIndex"]
      .apply(lambda s: sorted(s.tolist()) == list(range(1, len(s) + 1)))
)
bad_idx_instances = ok_idx[~ok_idx].index.tolist()
report["instances_bad_event_index"] = len(bad_idx_instances)
if bad_idx_instances:
    issues.append(f"Экземпляры с нарушением EventIndex 1..N: {len(bad_idx_instances)}")

# 3. Первое событие — process_started, последнее — process_completed
srt = df.sort_values(["ProcessInstanceId", "EventIndex"])
first_ev = srt.groupby("ProcessInstanceId").first()
last_ev  = srt.groupby("ProcessInstanceId").last()

bad_start = first_ev[first_ev["EventType"] != "process_started"].index.tolist()
bad_end   = last_ev [last_ev ["EventType"] != "process_completed"].index.tolist()
report["instances_bad_start"] = len(bad_start)
report["instances_bad_end"]   = len(bad_end)
if bad_start: issues.append(f"Нестандартный старт: {len(bad_start)} экз.")
if bad_end:   issues.append(f"Нестандартное окончание: {len(bad_end)} экз.")

# 4. task_completed: TaskId, UserId, Role, Result должны быть заполнены
mask_task = df["EventType"] == "task_completed"
missing_task = df[mask_task & (
    (df["TaskId"]  == "") |
    (df["UserId"]  == "") |
    (df["Role"]    == "") |
    (df["Result"]  == "")
)]
report["task_completed_rows"]          = int(mask_task.sum())
report["task_completed_missing_fields"] = len(missing_task)
if len(missing_task):
    issues.append(f"task_completed с пустыми обязательными полями: {len(missing_task)}")

# 5. system-события: Role=system, TaskId и UserId пустые
mask_sys = df["Role"] == "system"
bad_sys  = df[mask_sys & ((df["TaskId"] != "") | (df["UserId"] != ""))]
report["system_rows"]      = int(mask_sys.sum())
report["system_rows_bad"]  = len(bad_sys)
if len(bad_sys):
    issues.append(f"system-события с непустыми TaskId/UserId: {len(bad_sys)}")

# 6. ActivityId ↔ ActivityName — один-к-одному
aid_to_name = df.groupby("ActivityId")["ActivityName"].nunique()
name_to_aid = df.groupby("ActivityName")["ActivityId"].nunique()
report["activity_id_used_by_multiple_names"] = int((aid_to_name > 1).sum())
report["activity_name_used_by_multiple_ids"] = int((name_to_aid > 1).sum())
if (aid_to_name > 1).any():
    issues.append("Один ActivityId связан с несколькими ActivityName")
if (name_to_aid > 1).any():
    issues.append("Один ActivityName связан с несколькими ActivityId")

# 7. Справочники
report["event_types"]  = df["EventType"].value_counts().to_dict()
report["roles"]        = df["Role"].value_counts(dropna=False).to_dict()
report["activities"]   = df["ActivityName"].value_counts().to_dict()
report["results"]      = df["Result"].fillna("").value_counts().to_dict()

# 8. Итог
print("=== SUMMARY ===")
for k, v in report.items():
    print(f"{k:42s}: {v}")

print("\n=== ISSUES ===")
print("\n".join(issues) if issues else "OK: критичных проблем не найдено")

print("\n=== EVENT_TYPE x ROLE ===")
print(pd.crosstab(df["EventType"], df["Role"]))

Профиль активностей и экземпляров

In [ ]:
# Cell 3: per-instance and per-activity profile

# --- 3.1. Последовательность и границы каждого экземпляра ---
inst = (df.sort_values(["ProcessInstanceId", "EventIndex"])
          .groupby("ProcessInstanceId")
          .agg(
              first_activity = ("ActivityName", "first"),
              last_activity  = ("ActivityName", "last"),
              n_events       = ("EventId",      "size"),
              n_tasks        = ("EventType",    lambda s: (s == "task_completed").sum()),
              sequence       = ("ActivityName", lambda s: tuple(s)),
          )
          .reset_index())

# Итоговый Result и Amount процесса берём из process_completed
final = (df[df["EventType"] == "process_completed"]
         [["ProcessInstanceId", "Result", "Amount", "Timestamp"]]
         .rename(columns={"Result": "final_result",
                          "Amount": "final_amount",
                          "Timestamp": "finished_at"}))
inst = inst.merge(final, on="ProcessInstanceId", how="left")

# Проверка: Amount постоянна внутри экземпляра
amount_nuniq = df.groupby("ProcessInstanceId")["Amount"].nunique()
print("Экземпляров с 'плавающей' Amount:", int((amount_nuniq > 1).sum()))

# --- 3.2. Старт / финал ---
print("\nПервая активность по экземплярам:")
print(inst["first_activity"].value_counts())

print("\nПоследняя активность по экземплярам:")
print(inst["last_activity"].value_counts())

print("\nФинальный Result:")
print(inst["final_result"].value_counts(dropna=False))

# --- 3.3. Профиль по активностям: события, экземпляры, роли, результаты ---
act_rows = []
for name, g in df[df["EventType"] == "task_completed"].groupby("ActivityName"):
    roles    = g["Role"].value_counts().to_dict()
    results  = g["Result"].value_counts().to_dict()
    n_events = len(g)
    n_inst   = g["ProcessInstanceId"].nunique()
    act_rows.append({
        "ActivityName":  name,
        "Events":        n_events,
        "Instances":     n_inst,
        "ShareOfInst":   round(n_inst / inst.shape[0], 3),
        "Roles":         roles,
        "Results":       results,
    })
activity_profile = (pd.DataFrame(act_rows)
                      .sort_values("Events", ascending=False)
                      .reset_index(drop=True))
print("\nПрофиль активностей:")
print(activity_profile.to_string(index=False))

# --- 3.4. Повторы активностей внутри экземпляра ---
repeat = (df[df["EventType"] == "task_completed"]
            .groupby(["ProcessInstanceId", "ActivityName"]).size()
            .rename("occurrences").reset_index()
            .query("occurrences > 1"))
print("\nАктивности, встречающиеся >1 раза в экземпляре (по событиям):")
print(repeat.groupby("ActivityName")["ProcessInstanceId"].nunique()
            .sort_values(ascending=False))
print("\nВсего экземпляров с хотя бы одним повтором:",
      repeat["ProcessInstanceId"].nunique())

# --- 3.5. Кросс-таблица: финал × последняя задача ---
print("\nФинал × последняя task_completed:")
last_task = (df[df["EventType"] == "task_completed"]
               .sort_values(["ProcessInstanceId", "EventIndex"])
               .groupby("ProcessInstanceId").last()["ActivityName"]
               .rename("last_task"))
ct = (inst.set_index("ProcessInstanceId")
          .join(last_task)
          .groupby(["final_result", "last_task"]).size().unstack(fill_value=0))
print(ct)

# --- 3.6. Сохраняем профиль ---
activity_profile.to_csv(OUT_DIR / "activity_profile.csv", index=False)
inst[["ProcessInstanceId", "first_activity", "last_activity",
      "n_events", "n_tasks", "final_result", "final_amount"]].to_csv(
    OUT_DIR / "instance_profile.csv", index=False)

Переходы

In [ ]:
# Cell 4: transitions + graph + student_transitions.csv

df_s = df.sort_values(["ProcessInstanceId", "EventIndex"]).reset_index(drop=True)
df_s["NextActivityName"] = df_s.groupby("ProcessInstanceId")["ActivityName"].shift(-1)

pairs = df_s.dropna(subset=["NextActivityName"])

# --- 4.1. Сырые переходы ---
transitions = (pairs.groupby(["ActivityName", "NextActivityName"])
                    .size().rename("Count").reset_index()
                    .sort_values(["ActivityName", "Count"], ascending=[True, False]))
print("=== Переходы (соседние ActivityName) ===")
print(transitions.to_string(index=False))

# --- 4.2. Матрица переходов ---
print("\n=== Матрица переходов ===")
mat = (transitions.pivot(index="ActivityName", columns="NextActivityName",
                         values="Count").fillna(0).astype(int))
print(mat)

# --- 4.3. Санити-чек: сумма переходов = сумма (N_i - 1) ---
expected = int(df["ProcessInstanceId"].value_counts().sub(1).sum())
actual   = int(transitions["Count"].sum())
print(f"\nОжидаемо переходов (Σ(N-1)): {expected}")
print(f"Наблюдено переходов:         {actual}")
assert expected == actual, "Расхождение в сумме переходов!"

# --- 4.4. Переходы в разрезе Result ---
print("\n=== Переходы по Result ===")
res_trans = (pairs.groupby(["ActivityName", "Result", "NextActivityName"])
                  .size().rename("Count").reset_index()
                  .sort_values(["ActivityName", "Result", "Count"],
                               ascending=[True, True, False]))
print(res_trans.to_string(index=False))

# --- 4.5. Точки ветвления (узлы с >1 наблюдаемым преемником) ---
out_degree = transitions.groupby("ActivityName").size().rename("OutDegree")
decisions  = out_degree[out_degree > 1].index.tolist()
print("\n=== Точки ветвления ===")
for act in decisions:
    sub = transitions[transitions["ActivityName"] == act]
    print(f"\n{act}:")
    print(sub[["NextActivityName", "Count"]].to_string(index=False))

# --- 4.6. CSV переходов по шаблону ---
out_csv = transitions.rename(columns={"ActivityName":     "SourceActivity",
                                       "NextActivityName": "TargetActivity"})[
                ["SourceActivity", "TargetActivity"]]
out_csv.to_csv(OUT_DIR / "student_transitions.csv", index=False, encoding="utf-8")
print(f"\nСохранено {len(out_csv)} уникальных пар → student_transitions.csv")

# --- 4.7. Mermaid-граф ---
lines = ["flowchart LR"]
for _, r in transitions.iterrows():
    src = f'"{r["ActivityName"]}"'
    tgt = f'"{r["NextActivityName"]}"'
    lines.append(f"  {src} -->|{r['Count']}| {tgt}")
mermaid = "\n".join(lines)
(OUT_DIR / "graph_raw.mmd").write_text(mermaid, encoding="utf-8")
print("\n=== Mermaid (сырой граф) ===")
print(mermaid)

Условия развилок и полные варианты

In [ ]:
# Cell 5a: branching conditions + full variants

# --- 5.1. Классификация экземпляров по маршруту ---
inst["n_returns"] = inst["sequence"].apply(
    lambda s: sum(1 for x in s if x == "Доработка заявки"))
inst["has_dop"] = inst["sequence"].apply(
    lambda s: "Дополнительное согласование" in s)
inst["n_events"] = inst["n_events"]

print("=== n_events × n_returns × has_dop ===")
print(inst.groupby(["n_events", "n_returns", "has_dop"])
        .size().rename("Count").to_string())

# --- 5.2. Порог по Amount для доп. согласования ---
fulfilled = inst[inst["final_result"] == "fulfilled"]
wd = fulfilled[ fulfilled["has_dop"]]["final_amount"]
nd = fulfilled[~fulfilled["has_dop"]]["final_amount"]
print(f"\nС доп.:   n={len(wd):4d}  min={wd.min():10.2f}  max={wd.max():10.2f}")
print(f"Без доп.: n={len(nd):4d}  min={nd.min():10.2f}  max={nd.max():10.2f}")

if wd.min() > nd.max():
    print(f"→ Чёткий порог между {nd.max():.2f} и {wd.min():.2f}")
else:
    print(f"→ Пересечение (nd.max={nd.max():.2f} > wd.min={wd.min():.2f}) — условия не только Amount")

for thr in [100_000, 150_000, 200_000]:
    fp = int((wd <= thr).sum()); fn = int((nd > thr).sum())
    print(f"  thr={thr:>7d}: FP={fp:3d}, FN={fn:3d}")

# --- 5.3. Кто отменяет / отклоняет у менеджера — есть ли связь с Amount ---
mgr_first = (df[df["ActivityName"] == "Согласование руководителем"]
               .sort_values(["ProcessInstanceId", "EventIndex"])
               .groupby("ProcessInstanceId").first()[["Result"]]
               .rename(columns={"Result": "first_mgr_result"}))
inst2 = inst.set_index("ProcessInstanceId").join(mgr_first)
print("\n=== Amount по первому результату менеджера ===")
print(inst2.groupby("first_mgr_result")["final_amount"]
           .describe()[["count", "mean", "min", "50%", "max"]]
           .round(0).to_string())

# --- 5.4. Полные варианты ---
inst["route_str"] = inst["sequence"].apply(lambda s: " > ".join(s))
variants = (inst.groupby("route_str").size().rename("NumberOfCases")
              .reset_index().sort_values("NumberOfCases", ascending=False))
variants["Share"] = (variants["NumberOfCases"] / len(inst)).round(4)
variants.insert(0, "Variant", [f"V{i+1}" for i in range(len(variants))])
variants.to_csv(OUT_DIR / "variants.csv", index=False)

print("\n=== Полные варианты ===")
for _, r in variants.iterrows():
    print(f"{r['Variant']:>3s} ({r['NumberOfCases']:4d}, {r['Share']:.1%}): "
          f"{r['route_str']}")

# --- 5.5. Редкие комбинации: есть ли 2 возврата + доп.? ---
print("\n=== Комбинации возвратов × доп. ===")
comb = inst.groupby(["n_returns", "has_dop"]).size().rename("Count")
print(comb.to_string())

# --- 5.6. Сохраняем обогащённый профиль ---
inst[["ProcessInstanceId", "route_str", "n_events", "n_returns", "has_dop",
      "final_result", "final_amount"]].to_csv(
    OUT_DIR / "instances_with_route.csv", index=False)

In [ ]:
# Cell 5b: exhaustive threshold search + gap analysis

fulfilled = inst[inst["final_result"] == "fulfilled"]
wd = fulfilled[ fulfilled["has_dop"]]["final_amount"].values
nd = fulfilled[~fulfilled["has_dop"]]["final_amount"].values

lo, hi = nd.max(), wd.min()
print(f"С доп.:    n={len(wd)}, min={wd.min():.2f}, max={wd.max():.2f}")
print(f"Без доп.:  n={len(nd)}, min={nd.min():.2f}, max={nd.max():.2f}")
print(f"\nОткрытый зазор (nd.max; wd.min) = ({lo:.2f}; {hi:.2f})  "
      f"ширина={hi-lo:.2f} руб.")
n_in_gap = int(((fulfilled["final_amount"] > lo) &
                (fulfilled["final_amount"] <  hi)).sum())
print(f"Экземпляров строго внутри зазора: {n_in_gap}")

# Правило "Amount > thr  → доп.согласование"
# Условие FP=FN=0:  thr >= nd.max  И  thr < wd.min  ⇒  thr ∈ [lo; hi)
lo_gt, hi_gt = lo, hi           # полуоткрытый интервал [149743.41; 150614.59)
print(f"\nПравило '>': thr ∈ [{lo_gt:.2f}; {hi_gt:.2f})")

# Правило "Amount >= thr → доп.согласование"
# Условие FP=FN=0:  thr > nd.max  И  thr <= wd.min ⇒  thr ∈ (lo; hi]
lo_ge, hi_ge = lo, hi
print(f"Правило '>=': thr ∈ ({lo_ge:.2f}; {hi_ge:.2f}]")

# Явная проверка 150000 и других круглых значений
print("\nЯвная проверка кандидатов:")
for t in [100_000, 149_743.41, 149_750, 150_000, 150_500, 150_614.59, 200_000]:
    fp_gt = int((wd <= t).sum()); fn_gt = int((nd >  t).sum())
    fp_ge = int((wd <  t).sum()); fn_ge = int((nd >= t).sum())
    print(f"  thr={t:>10.2f}  '>': FP={fp_gt:3d} FN={fn_gt:3d}  "
          f"'>= ': FP={fp_ge:3d} FN={fn_ge:3d}")

# 150000 находится в интервале допустимых порогов?
t = 150_000
print(f"\n150000 ∈ [{lo:.2f}; {hi:.2f}) для правила '>'? {lo <= t < hi}")
print(f"150000 ∈ ({lo:.2f}; {hi:.2f}] для правила '>='? {lo < t <= hi}")

Сравнение с автоматической моделью

In [ ]:
# Cell 6a: Process Mining analysis 

import pm4py
from pm4py.objects.conversion.log import converter as log_converter 
from pm4py.objects.process_tree.obj import Operator

# 1. Подготовка лога
ev = df.rename(columns={
    "ProcessInstanceId": "case:concept:name",
    "ActivityName":      "concept:name",
    "Timestamp":         "time:timestamp",
    "UserId":            "org:resource",
})[["case:concept:name", "concept:name", "time:timestamp", "org:resource"]]
log = log_converter.apply(ev, variant=log_converter.Variants.TO_EVENT_LOG)
n_events = sum(len(trace) for trace in log)
print(f"Событий в данных: {n_events}, экземпляров: {len(log)}")

# 2. DFG
dfg, sa, ea = pm4py.discover_dfg(log)
print(f"\n=== DFG: {len(dfg)} рёбер ===")
for (a, b), c in sorted(dfg.items(), key=lambda x: -x[1]):
    print(f"  {a:>32s} → {b:<32s} {c}")

# 3. Inductive Miner
net, im, fm = pm4py.discover_petri_net_inductive(log)
print(f"\nPetri net: {len(net.places)} мест, {len(net.transitions)} переходов")

def print_tree(node, depth=0):
    indent = "  " * depth
    if node.operator is not None:
        op = {Operator.SEQUENCE: "->", Operator.XOR: "X",
              Operator.LOOP: "*", Operator.PARALLEL: "+",
              Operator.OR: "O"}.get(node.operator, str(node.operator))
        print(f"{indent}{op}")
        for child in node.children:
            print_tree(child, depth + 1)
    else:
        print(f"{indent}{node.label}")

tree = pm4py.discover_process_tree_inductive(log)
print_tree(tree)

# 4. Fitness
fitness = pm4py.fitness_token_based_replay(log, net, im, fm)
print("\nToken-based fitness:", fitness)

# 5. Precision 
precision = pm4py.precision_token_based_replay(log, net, im, fm)
print("Precision (ETConformance):", precision)




In [ ]:
# Cell 6b: Decision Mining — какие атрибуты объясняют выбор ветки?
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text

# ---  Проверка бюджета → доп. согласование? ---
budget_cases = inst[inst["sequence"].apply(
    lambda s: "Проверка бюджета" in s
)].copy()
budget_cases["went_dop"] = budget_cases["has_dop"].astype(int)

X1 = budget_cases[["final_amount"]].values
y1 = budget_cases["went_dop"].values

tree1 = DecisionTreeClassifier(max_depth=1, random_state=42).fit(X1, y1)
thr1  = tree1.tree_.threshold[0]
acc1  = tree1.score(X1, y1)

print("=== Проверка бюджета ===")
print(f"Порог из дерева:              {thr1:.2f}")
print(f"Accuracy на обучающей выборке: {acc1:.4f}")
print(export_text(tree1, feature_names=["Amount"]))
print(f"Интервал допустимых порогов: (149743.41; 150614.59)")
print(f"Порог дерева внутри интервала: {149743.41 < thr1 < 150614.59}")

# --- Согласование → какая ветка? ---
mgr_first = (df[df["ActivityName"] == "Согласование руководителем"]
               .sort_values(["ProcessInstanceId", "EventIndex"])
               .groupby("ProcessInstanceId").first()[["Result"]]
               .rename(columns={"Result": "mgr_result"}))
mgr_cases = (inst.set_index("ProcessInstanceId")
                  .join(mgr_first, how="inner").reset_index())

X2 = mgr_cases[["final_amount"]].values
y2 = mgr_cases["mgr_result"].values

tree2 = DecisionTreeClassifier(max_depth=1, random_state=42).fit(X2, y2)
acc2  = tree2.score(X2, y2)
thr2  = tree2.tree_.threshold[0]

print("\n=== Согласование руководителем ===")
print(f"Accuracy на обучающей выборке: {acc2:.4f}")
print(f"Единственный разрез по Amount: {thr2:.2f}")
print(export_text(tree2, feature_names=["Amount"]))
print(f"\nРаспределение классов: {mgr_cases['mgr_result'].value_counts().to_dict()}")

# Сводка 
summary = pd.DataFrame([
    {"Decision": "Проверка бюджета → доп. согласование?",
     "Attribute": "Amount", "Depth": 1,
     "Threshold": round(thr1, 2), "Accuracy": round(acc1, 4)},
    {"Decision": "Согласование → accept/return/reject/cancel?",
     "Attribute": "Amount", "Depth": 1,
     "Threshold": round(thr2, 2), "Accuracy": round(acc2, 4)},
])
print("\n=== Сводка ===")
print(summary.to_string(index=False))
